# Lesson 11: k-nearest neighbours and the next US recession

1. How does kNN decide whether a recession follows a month?
2. What makes two months near each other?
3. How many neighbours should vote?
4. How is k chosen and the model judged?

Monthly US data, April 1953 to September 2026: the term spread, the 10-year Treasury yield minus the 3-month bill rate; the T-bill change, the 3-month bill rate minus its value 12 months earlier; and the NBER recession months. A month counts as followed by a recession month 12 months later when the month 12 months later is an NBER recession month; the label ends in September 2024. Training months run to December 1999, test months from January 2000.

Treasury yields and recession dates: Federal Reserve Board H.15 and NBER via FRED, Federal Reserve Bank of St. Louis. Snapshot 30 September 2026.

In [ ]:
import pathlib

import pandas as pd

DATA_URL = "https://digital-ai-finance.github.io/data-science-msc/data/"
SNAPSHOT = "2026-09-30"


def load(name):
    """Read a snapshot: "us-recession-monthly"."""
    local = pathlib.Path("../data/market") / f"{SNAPSHOT}-{name}.csv"
    if local.exists():  # run from material/notebooks/ before the files are published
        return pd.read_csv(local)
    url = f"{DATA_URL}{SNAPSHOT}-{name}.csv"
    try:
        return pd.read_csv(url)
    except Exception as err:
        raise RuntimeError(f"{url} not reachable ({err}). Download the file in a browser, upload "
                           "it in the Colab file panel and read it with pd.read_csv.") from err

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FEATURES = ["spread", "tbill_change_12m"]
TARGET = "recession_in_12m"
months = load("us-recession-monthly")
train = months[months["split"] == "train"]
test = months[months["split"] == "test"]
future = months[months["split"] == "unlabelled"]
assert train["month"].max() < test["month"].min()  # every training month lies before every test month
X_train, y_train = train[FEATURES].to_numpy(float), train[TARGET].astype(int).to_numpy()
X_test, y_test = test[FEATURES].to_numpy(float), test[TARGET].astype(int).to_numpy()
assert (len(train), len(test), y_train.sum(), y_test.sum()) == (561, 297, 77, 28)
print("training", X_train.shape, "| test", X_test.shape, "| future", len(future), "months")
print(f"share followed by a recession month 12 months later: training {y_train.mean():.3f}, test {y_test.mean():.3f}")

## 1. How does kNN decide whether a recession follows a month?

kNN stores the training months. For a new month it measures the distance to every training month, takes the k nearest and lets them vote: the share of them followed by a recession month 12 months later is the predicted probability. July 2000 is a test month, and the recession that began in April 2001 followed it.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train)  # means and standard deviations of the training months only
Z_train = scaler.transform(X_train)
july = months.loc[months["month"] == "2000-07", FEATURES].to_numpy(float)
z_july = scaler.transform(july)[0]
distance = np.sqrt(((Z_train - z_july) ** 2).sum(axis=1))
nearest = np.argsort(distance, kind="stable")[:7]
for rank, i in enumerate(nearest, start=1):
    print(f"{rank}  {train['month'].iloc[i]}  distance {distance[i]:.3f}  recession 12 months later: {y_train[i]}")
for k in (3, 5, 7):
    vote = y_train[nearest[:k]].mean()
    print(f"k = {k}: vote share {vote:.2f}, prediction {'recession' if vote > 0.5 else 'no recession'}")

In [ ]:
print(f"distances in one pass over the test months: {len(X_test)} x {len(X_train)} = {len(X_test) * len(X_train):,}")

## 2. What makes two months near each other?

The Euclidean distance adds the squared differences of the features and takes the root. Both features are in percentage points, and the unit of a feature sets its weight in the distance. Standardising subtracts the training mean of each feature and divides by its training standard deviation.

In [ ]:
row = months.set_index("month")[FEATURES]
q, a, b = (row.loc[m].to_numpy(float) for m in ("2000-07", "1968-04", "1969-10"))
sd = scaler.scale_


def euclid(u, v, weight=np.ones(2)):
    """Euclidean distance after multiplying each feature difference by its weight."""
    return float(np.sqrt((((u - v) * weight) ** 2).sum()))


print("Q July 2000", q, "| A April 1968", a, "| B October 1969", b)
print(f"raw points:            d(Q, A) = {euclid(q, a):.2f}   d(Q, B) = {euclid(q, b):.2f}")
print(f"spread in basis points: d(Q, A) = {euclid(q, a, np.array([100, 1])):.1f}   d(Q, B) = {euclid(q, b, np.array([100, 1])):.2f}")
print(f"standardised:          d(Q, A) = {euclid(q, a, 1 / sd):.2f}   d(Q, B) = {euclid(q, b, 1 / sd):.2f}")
print(f"A with spread 0.20, standardised: d(Q, A) = {euclid(q, np.array([0.20, a[1]]), 1 / sd):.2f}")

In [ ]:
print(f"training means: spread {scaler.mean_[0]:.2f}, T-bill change {scaler.mean_[1]:.2f}")
print(f"training standard deviations: spread {scaler.scale_[0]:.2f}, T-bill change {scaler.scale_[1]:.2f}")
Z_test = scaler.transform(X_test)  # the test months on the training scale
print(f"means of the standardised test columns: spread {Z_test[:, 0].mean():.3f}, T-bill change {Z_test[:, 1].mean():.3f}")
own = (X_test[:, 0] - X_test[:, 0].mean()) / scaler.scale_[0]
print(f"test mean spread {X_test[:, 0].mean():.2f}; with it, each standardised test spread falls by {(Z_test[:, 0] - own).mean():.2f}")
print(f"a T-bill change of +4.00, standardised: {(4.00 - scaler.mean_[1]) / scaler.scale_[1]:.2f}")

## 3. How many neighbours should vote?

Few neighbours follow single months; many neighbours smooth the vote until every month gets the same share. Three forward folds of the training months score each k by its AUC: each fold trains on the months before a block and scores the block. A month's label looks 12 months ahead, so `gap=12` leaves the last 12 months before each block out of its training. `K_GRID` stops at 101 and holds odd k only, so a vote between two classes never ties.

In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neighbors import KNeighborsClassifier

K_GRID = [1, 3, 5, 9, 15, 25, 51, 101]
folds = TimeSeriesSplit(n_splits=3, gap=12)  # a label looks 12 months ahead
for fit_idx, held_idx in folds.split(X_train):
    print(f"{len(fit_idx)} training months; held out {train['month'].iloc[held_idx[0]]} to "
          f"{train['month'].iloc[held_idx[-1]]}, {y_train[held_idx].sum()} of them followed by a recession")
rows = []
for k in K_GRID:
    fit_auc, held_auc = [], []
    for fit_idx, held_idx in folds.split(X_train):
        scaler = StandardScaler().fit(X_train[fit_idx])
        model = KNeighborsClassifier(n_neighbors=k).fit(scaler.transform(X_train[fit_idx]), y_train[fit_idx])
        fit_share = model.predict_proba(scaler.transform(X_train[fit_idx]))[:, 1]
        held_share = model.predict_proba(scaler.transform(X_train[held_idx]))[:, 1]
        fit_auc.append(roc_auc_score(y_train[fit_idx], fit_share))
        held_auc.append(roc_auc_score(y_train[held_idx], held_share))
    rows.append({"k": k, "training": np.mean(fit_auc), "held_out": np.mean(held_auc), "held_out_sd": np.std(held_auc)})
cv = pd.DataFrame(rows)
print(cv.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

In [ ]:
fit_idx, held_idx = next(folds.split(X_train))
sc = StandardScaler().fit(X_train[fit_idx])
every = KNeighborsClassifier(n_neighbors=len(fit_idx)).fit(sc.transform(X_train[fit_idx]), y_train[fit_idx])
auc_every = roc_auc_score(y_train[held_idx], every.predict_proba(sc.transform(X_train[held_idx]))[:, 1])
print(f"first fold, k = {len(fit_idx)}, all its training months voting: held-out AUC {auc_every:.3f}")
plt.plot(cv["k"], cv["training"], "o-", color="darkgray", label="training")
plt.plot(cv["k"], cv["held_out"], "o-", color="black", label="held out")
plt.xscale("log")
plt.xticks(K_GRID, K_GRID)
plt.xlabel("k, number of neighbours")
plt.ylabel("mean AUC over three forward folds")
plt.legend()
plt.show()

In [ ]:
from matplotlib.colors import ListedColormap


def plot_decision_boundary(ax, X, y, k, title):
    """Shade the majority vote of k neighbours over a grid around the months X."""
    h = 0.05
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    knn = KNeighborsClassifier(n_neighbors=k).fit(X, y)
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.4, cmap=ListedColormap(["#cce0ff", "#ffd0d0"]))
    ax.scatter(X[:, 0], X[:, 1], c=np.where(y == 1, "tab:red", "tab:blue"), s=8)
    ax.set_title(title)
    ax.set_xlabel("spread (standardised)")
    ax.set_ylabel("T-bill change (standardised)")


scaler = StandardScaler().fit(X_train)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, k in zip(axes, [1, 15]):
    plot_decision_boundary(ax, scaler.transform(X_train), y_train, k, f"k = {k}; red: recession 12 months later")
plt.tight_layout()
plt.show()

## 4. How is k chosen and the model judged?

The forward folds pick the k with the highest mean held-out AUC. The model is then fitted on all 561 training months and scored once on the 297 test months, 2000 to 2024. The model warns at a vote share of 0.20 or more, the threshold of Lesson 10.

In [ ]:
cv["rounded"] = cv["held_out"].round(4)
best_k = int(cv.sort_values(["rounded", "k"], ascending=[False, True])["k"].iloc[0])
scaler = StandardScaler().fit(X_train)
model = KNeighborsClassifier(n_neighbors=best_k).fit(scaler.transform(X_train), y_train)
share = model.predict_proba(scaler.transform(X_test))[:, 1]
print(f"k = {best_k}; distances computed: {len(X_test) * len(X_train):,}")
print(f"test AUC: {roc_auc_score(y_test, share):.3f}")
print(f"warnings at 0.5: {(share > 0.5).sum()}, accuracy {((share > 0.5) == y_test).mean():.3f}; "
      f"predicting no recession for every month: {(y_test == 0).mean():.3f}")
tn, fp, fn, tp = confusion_matrix(y_test, (share >= 0.20).astype(int), labels=[0, 1]).ravel()
print(f"at 0.20: TP {tp}, FP {fp}, FN {fn}, TN {tn}; {tp + fp} warnings; recall {tp / (tp + fn):.2f}, precision {tp / (tp + fp):.2f}")
cm = np.array([[tp, fn], [fp, tn]])  # rows: recession, none; columns: warning, no warning
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=16, fontweight="bold",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xticks([0, 1], ["warning", "no warning"])
ax.set_yticks([0, 1], ["recession 12 months later", "none"])
ax.set_title(f"k = {best_k}, threshold 0.20, {len(y_test)} test months")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

labelled = months[months["split"] != "unlabelled"]
X_all, y_all = labelled[FEATURES].to_numpy(float), labelled[TARGET].astype(int).to_numpy()
fit_idx, held_idx = train_test_split(np.arange(len(labelled)), test_size=0.2, random_state=42)
rnd_scaler = StandardScaler().fit(X_all[fit_idx])
rnd_model = KNeighborsClassifier(n_neighbors=best_k).fit(rnd_scaler.transform(X_all[fit_idx]), y_all[fit_idx])
rnd_share = rnd_model.predict_proba(rnd_scaler.transform(X_all[held_idx]))[:, 1]
print(f"random 80/20 split, {len(fit_idx)} and {len(held_idx)} months: AUC {roc_auc_score(y_all[held_idx], rnd_share):.3f}")
print(f"split by date, {len(X_train)} and {len(X_test)} months: AUC {roc_auc_score(y_test, share):.3f}")
held_pos = held_idx[y_all[held_idx] == 1]
fit_pos = set(fit_idx[y_all[fit_idx] == 1])
near = sum(any(j in fit_pos for j in range(i - 6, i + 7) if j != i) for i in held_pos)
print(f"random test months followed by a recession: {len(held_pos)}; with a training month of the same "
      f"recession within six months: {near}")

In [ ]:
future_share = model.predict_proba(scaler.transform(future[FEATURES].to_numpy(float)))[:, 1]
plt.plot(pd.to_datetime(future["month"]), future_share, "o-", color="black")
plt.axhline(0.20, ls="--", color="tab:orange", label="threshold 0.20")
plt.ylabel(f"vote share, k = {best_k}")
plt.legend()
plt.show()
print(f"{future['month'].iloc[-1]}: vote share {future_share[-1]:.2f}, a forecast whose outcome is unknown")

## Exercises

**E1.** Add the 10-year yield `gs10` as a third feature, rerun the forward folds and compare the chosen k and its held-out AUC.

**E2.** Run the forward folds with `KNeighborsClassifier(n_neighbors=k, metric="manhattan")` and compare the chosen k.

**E3.** Replace `TimeSeriesSplit` by `KFold(n_splits=3, shuffle=True, random_state=42)` in the fold loop and compare the held-out AUC of every k with the forward folds.

**E4.** Fit `LogisticRegression` on the spread alone, as in Lesson 10, and compare its test AUC with the 0.688 of kNN.

## The k nearest training months vote on a recession, and forward folds pick k

1. The k nearest of all 561 training months vote; the share with a recession month 12 months later is the probability p.
2. A small Euclidean distance over features standardised with training values, each known by the end of its month.
3. Few neighbours draw a decision boundary around single months and overfit; all training months give one share. Held-out AUC peaks at k = 101.
4. Forward folds with a 12-month gap pick k; the test months 2000 to 2024, used once, give AUC 0.688, below the 0.752 of the logistic regression on the spread.

## Newer data (optional)

The cell below fetches the live series from FRED. Its numbers differ from the slides, which use the snapshot of 30 September 2026.

In [ ]:
FRED = "https://fred.stlouisfed.org/graph/fredgraph.csv?id="


def fred(*ids):
    """Live FRED series, e.g. fred("GS10", "TB3MS", "USREC"); "." marks a missing value."""
    return pd.read_csv(FRED + ",".join(ids), na_values=["."], parse_dates=["observation_date"])


live = fred("GS10", "TB3MS").dropna().set_index("observation_date")
live["spread"] = live["GS10"] - live["TB3MS"]
live["tbill_change_12m"] = live["TB3MS"] - live["TB3MS"].shift(12)
latest = live.dropna().iloc[[-1]]
print(latest.index[0].date(), latest[FEATURES].round(2).to_dict("records")[0])
print(f"vote share, k = {best_k}: {model.predict_proba(scaler.transform(latest[FEATURES].to_numpy(float)))[0, 1]:.2f}")